# 📚 Scraper Putusan JDIH Komisi Informasi Pusat

Notebook ini mengambil data **putusan** dari <https://jdih.komisiinformasi.go.id/putusan> (±904 putusan)
dan menyimpannya ke **Excel + CSV**. Kalau mau, PDF putusan juga bisa diunduh dan teksnya diekstrak.

**Tidak perlu menulis kode.** Jalankan sel dari atas ke bawah. Kalau ada pilihan (kotak centang, dropdown),
atur dulu, *lalu* jalankan sel berikutnya.

| Langkah | Isi | Wajib? |
|---|---|---|
| 1 | Pasang paket | ✅ |
| 1b | Pasang browser otomatis | hanya kalau diminta di Langkah 3 |
| 2 | Pengaturan | ✅ (boleh biarkan bawaan) |
| 3 | Ambil **daftar** semua putusan | ✅ |
| 4 | **Saring**: tahun, jenis, kata kunci, jumlah | ✅ (bawaan = semua) |
| 5 | Pratinjau 3 putusan → **pilih kolom** yang mau disimpan | ✅ |
| 6 | Scrape detail semua putusan terpilih | ✅ |
| 7 | Simpan ke Excel/CSV | ✅ |
| 8 | Unduh PDF / ekstrak teks PDF | opsional |
| 9 | Kemas jadi ZIP & unduh | opsional |

### Cara menjalankan sel
- **Google Colab** (paling mudah): *File → Upload notebook*, lalu klik ▶ di kiri tiap sel (atau `Shift+Enter`).
- **Jupyter / VS Code / Anaconda**: buka file ini, jalankan sel satu per satu dengan `Shift+Enter`.
- Mau langsung semua dengan pilihan bawaan (ambil **semua** putusan, tanpa PDF)? *Runtime/Run → Run all*.

### Catatan
- Scraper sengaja diberi **jeda antar-request** (bawaan 1 detik) supaya tidak membebani server pemerintah.
  ±904 putusan ≈ 20–40 menit.
- Kalau terputus (internet mati, Colab disconnect), **jalankan ulang saja** — yang sudah diambil disimpan di
  cache dan tidak diambil ulang.
- Notebook ini **membaca sendiri struktur situs**: mencoba HTML biasa dulu, lalu (kalau situsnya dibangun
  dengan JavaScript) membuka situs pakai browser otomatis dan memakai API JSON-nya bila ada.

## Langkah 1 — Pasang paket
Cukup sekali per sesi. Di Colab butuh ±30 detik.

In [ ]:
%pip install -q requests beautifulsoup4 lxml pandas openpyxl tqdm ipywidgets pypdf
print("✅ Paket terpasang.")

## Langkah 1b — Pasang browser otomatis (opsional)
**Lewati langkah ini dulu.** Jalankan hanya kalau Langkah 3 atau 6 menampilkan pesan *"Jalankan Langkah 1b"*
(artinya situs dibangun dengan JavaScript sehingga perlu dibuka pakai browser). Butuh 1–3 menit.
Setelah selesai, ulangi langkah yang tadi meminta (Langkah 3 atau 6).

In [ ]:
import sys
%pip install -q playwright
!{sys.executable} -m playwright install --with-deps chromium
print("✅ Browser siap.")

## Langkah 0 — Muat mesin scraper
Jalankan sel ini (isinya kode, tidak perlu dibaca). Harus muncul **✅ Mesin scraper siap**.

In [ ]:
#@title ⚙️ Mesin scraper — cukup dijalankan, tidak perlu dibaca / diubah
import os, re, json, time, random, html as _html, warnings, concurrent.futures as _cf
from pathlib import Path
from urllib.parse import urljoin, urlparse, urlencode, parse_qs, unquote

import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
try:
    from tqdm.auto import tqdm
except ImportError:  # tanpa tqdm tetap jalan, hanya tanpa progress bar
    def tqdm(x=None, **k):
        return x

BASE_URL = os.environ.get("JDIH_BASE_URL", "https://jdih.komisiinformasi.go.id").rstrip("/")
LIST_PATH = "/putusan"
PDF_DIR_GUESS = "/api/uploads/dokumen/"   # pola link PDF yang terlihat di situs
UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")


class Config:
    delay = 1.0          # jeda antar-request (detik)
    timeout = 40
    out_dir = Path("hasil_scraping")

    @property
    def cache_dir(self):
        p = self.out_dir / "cache_detail"
        p.mkdir(parents=True, exist_ok=True)
        return p


CFG = Config()


# ----------------------------------------------------------------- HTTP ----
def _make_session():
    s = requests.Session()
    s.headers.update({"User-Agent": UA, "Accept-Language": "id-ID,id;q=0.9,en;q=0.8"})
    retry = Retry(total=4, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET", "HEAD"])
    s.mount("http://", HTTPAdapter(max_retries=retry))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


SESSION = _make_session()
_last_req = [0.0]


def fetch(url, **kw):
    """GET dengan jeda sopan + retry. Kalau sertifikat SSL situs bermasalah, otomatis lanjut tanpa verifikasi."""
    wait = CFG.delay - (time.time() - _last_req[0])
    if wait > 0:
        time.sleep(wait + random.uniform(0, CFG.delay * 0.3))
    try:
        r = SESSION.get(url, timeout=CFG.timeout, **kw)
    except requests.exceptions.SSLError:
        if SESSION.verify is False:
            raise
        print("⚠️  Sertifikat SSL situs bermasalah — lanjut tanpa verifikasi SSL (data publik, aman untuk dibaca).")
        SESSION.verify = False
        warnings.filterwarnings("ignore", message="Unverified HTTPS request")
        r = SESSION.get(url, timeout=CFG.timeout, **kw)
    finally:
        _last_req[0] = time.time()
    r.raise_for_status()
    return r


# -------------------------------------------------------------- helpers ----
def clean(t):
    if t is None:
        return ""
    return re.sub(r"\s+", " ", _html.unescape(str(t))).strip()


_HOST = urlparse(BASE_URL).netloc


def _same_host(u):
    return urlparse(u).netloc in ("", _HOST)


_DETAIL_RE = re.compile(r"^/putusan/([^/?#]+)/?$", re.I)
_NOT_DETAIL = {"page", "search", "cari", "kategori", "category", "tag", "tahun", "index", "list", "filter"}


def slug_of(url):
    m = _DETAIL_RE.match(urlparse(url).path)
    if not m:
        return None
    s = unquote(m.group(1))
    if s.lower() in _NOT_DETAIL or s.isdigit():
        return None
    return s


def detail_url(slug):
    return f"{BASE_URL}{LIST_PATH}/{slug}"


def _abs_pdf(v, page_url=None):
    v = str(v).strip()
    if not v:
        return None
    if v.startswith(("http://", "https://")):
        return v
    if v.startswith("/"):
        return BASE_URL + v
    if "/" in v and page_url:
        return urljoin(page_url, v)
    return BASE_URL + PDF_DIR_GUESS + v


def safe_name(s, maxlen=120):
    s = re.sub(r"[\\/:*?\"<>|\s]+", "_", str(s)).strip("._")
    return s[:maxlen] or "dokumen"


# ---------------------------------------------- informasi dari judul -------
_NOMOR_RE = re.compile(r"\b(\d{1,4}\s*/\s*[IVXLC]{1,5}\s*/?\s*KI[A-Z0-9\-\.]*\s*/\s*(?:19|20)\d{2})", re.I)


def derive_from_title(title, slug=""):
    """Ambil nomor, jenis, tahun, pemohon & termohon dari judul (contoh: 'Putusan ... Nomor 001/I/KIP-PSI-A/2025 Tahun 2025 A vs B')."""
    t = clean(title) or clean(slug.replace("-", " "))
    out = {"jenis_dokumen": "", "nomor_putusan": "", "tahun": "", "pemohon": "", "termohon": ""}
    low = t.lower()
    for key, lab in [("putusan mediasi", "Putusan Mediasi"), ("putusan sela", "Putusan Sela"),
                     ("penetapan", "Penetapan"), ("putusan", "Putusan")]:
        if low.startswith(key) or f" {key} " in f" {low[:40]} ":
            out["jenis_dokumen"] = lab
            break
    m = _NOMOR_RE.search(t)
    if m:
        out["nomor_putusan"] = re.sub(r"\s+", "", m.group(1)).upper()
    m = re.search(r"\btahun\s+((?:19|20)\d{2})\b", t, re.I)
    if m:
        out["tahun"] = m.group(1)
    elif out["nomor_putusan"]:
        out["tahun"] = out["nomor_putusan"][-4:]
    else:
        m = re.search(r"tahun-((?:19|20)\d{2})", slug) or re.search(r"((?:19|20)\d{2})", slug)
        if m:
            out["tahun"] = m.group(1)
    m = re.search(r"\s(?:vs\.?|versus|v\.|melawan)\s", t, re.I)
    if m:
        left, right = t[:m.start()], t[m.end():]
        cut = 0
        for pat in (r"\btahun\s+(?:19|20)\d{2}\b", _NOMOR_RE.pattern):
            for mm in re.finditer(pat, left, re.I):
                cut = max(cut, mm.end())
        out["pemohon"] = left[cut:].strip(" -:,")
        out["termohon"] = right.strip(" -:,")
    return out


def info_from_slug(slug):
    s = slug.lower()
    jenis = ""
    for key, lab in [("putusan-mediasi", "Putusan Mediasi"), ("penetapan", "Penetapan"), ("putusan", "Putusan")]:
        if s.startswith(key):
            jenis = lab
            break
    m = re.search(r"tahun-((?:19|20)\d{2})", s) or re.search(r"((?:19|20)\d{2})", s)
    return {"jenis_slug": jenis or "Lainnya", "tahun_slug": m.group(1) if m else "?"}


# ------------------------------------------------------- JSON utilities ----
_HINTS = {"slug", "judul", "title", "nomor", "no", "nomor_putusan", "no_putusan", "tahun", "file", "dokumen",
          "pemohon", "termohon", "tanggal", "jenis", "nama", "url", "lampiran", "abstrak"}


def _hint_score(d):
    return sum(1 for k in d if str(k).lower() in _HINTS or any(h in str(k).lower() for h in ("judul", "nomor", "slug", "file")))


def find_record_lists(obj, out=None):
    if out is None:
        out = []
    if isinstance(obj, list):
        dicts = [x for x in obj if isinstance(x, dict)]
        if dicts and len(dicts) >= max(1, len(obj) // 2):
            out.append((sum(_hint_score(d) for d in dicts[:5]) / min(5, len(dicts)), len(dicts), dicts))
        for x in obj[:50]:
            find_record_lists(x, out)
    elif isinstance(obj, dict):
        for v in obj.values():
            find_record_lists(v, out)
    return out


def best_records(obj):
    cands = [c for c in find_record_lists(obj) if c[0] >= 1]
    if not cands:
        return []
    cands.sort(key=lambda c: (c[0] >= 2, c[1], c[0]), reverse=True)
    return cands[0][2]


def best_dict(obj, slug=None):
    """Cari objek dict yang paling mirip 'data satu putusan' di dalam JSON."""
    best, best_score = None, 0
    stack = [obj]
    while stack:
        o = stack.pop()
        if isinstance(o, dict):
            sc = _hint_score(o)
            if slug and any(isinstance(v, str) and v == slug for v in o.values()):
                sc += 5
            if sc > best_score:
                best, best_score = o, sc
            stack.extend(o.values())
        elif isinstance(o, list):
            stack.extend(o[:200])
    return best if best_score >= 2 else None


def flatten(d, prefix="", out=None, depth=0):
    if out is None:
        out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict) and depth < 3:
            flatten(v, key + ".", out, depth + 1)
        elif isinstance(v, list):
            if all(not isinstance(x, (dict, list)) for x in v):
                out[key] = "; ".join(clean(x) for x in v)
            elif depth < 2 and len(v) <= 5:
                for i, x in enumerate(v, 1):
                    if isinstance(x, dict):
                        flatten(x, f"{key}.{i}.", out, depth + 1)
            else:
                out[key] = json.dumps(v, ensure_ascii=False)[:2000]
        else:
            out[key] = "" if v is None else (v if isinstance(v, (int, float, bool)) else clean(v))
    return out


def record_to_item(rec):
    flat = flatten(rec)
    slug = next((str(v) for k, v in flat.items() if k.lower().endswith("slug") and v), None)
    url = detail_url(slug) if slug else None
    if not url:
        for v in flat.values():
            if isinstance(v, str) and slug_of(urljoin(BASE_URL, v)):
                url = urljoin(BASE_URL, v)
                slug = slug_of(url)
                break
    judul = next((str(v) for k, v in flat.items()
                  if k.split(".")[-1].lower() in ("judul", "title", "nama", "name") and v), "")
    pdfs = []
    for v in flat.values():
        if isinstance(v, str) and re.search(r"\.pdf($|\?)", v, re.I):
            u = _abs_pdf(v)
            if u and u not in pdfs:
                pdfs.append(u)
    item = {"slug": slug or safe_name(judul)[:80], "url": url or "", "judul_daftar": judul,
            "pdf_dari_daftar": " | ".join(pdfs)}
    item.update({f"api.{k}": v for k, v in flat.items()})
    return item


# ------------------------------------------------------------- listing ----
def links_from_html(html, page_url):
    soup = BeautifulSoup(html, "lxml")
    found = {}
    for a in soup.find_all("a", href=True):
        u = urljoin(page_url, a["href"]).split("#")[0].split("?")[0]
        if not _same_host(u):
            continue
        sl = slug_of(u)
        if not sl:
            continue
        txt = clean(a.get("title") or a.get_text(" "))
        if sl not in found:
            found[sl] = {"slug": sl, "url": u, "judul_daftar": txt}
        elif len(txt) > len(found[sl]["judul_daftar"]):
            found[sl]["judul_daftar"] = txt
    return list(found.values())


def detect_total(html):
    """Perkiraan jumlah total putusan yang disebut di halaman (hanya untuk info/peringatan)."""
    text = clean(BeautifulSoup(html, "lxml").get_text(" "))
    nums = []
    for pat in (r"(?:dari|of|total|jumlah|sebanyak)\s*:?\s*([\d\.,]{1,7})\b",):
        for m in re.finditer(pat, text, re.I | re.M):
            n = int(re.sub(r"[\.,]", "", m.group(1)) or 0)
            if 20 <= n <= 100000 and not (1990 <= n <= 2040):  # angka tahun bukan jumlah
                nums.append(n)
    return max(nums) if nums else None


_PAGE_KEYS = ("page", "p", "hal", "halaman", "pg")


def detect_page_style(html, page_url):
    soup = BeautifulSoup(html, "lxml")
    for a in soup.find_all("a", href=True):
        u = urljoin(page_url, a["href"])
        pr = urlparse(u)
        if not _same_host(u):
            continue
        m = re.search(r"/putusan/page/(\d+)", pr.path)
        if m:
            return ("path", None, {})
        if pr.path.rstrip("/") != LIST_PATH:
            continue
        q = parse_qs(pr.query)
        for k, v in q.items():
            if k.lower() in _PAGE_KEYS and v and v[0].isdigit():
                return ("query", k, {kk: vv[0] for kk, vv in q.items() if kk != k})
    return ("query", "page", {})


def page_url(n, style):
    kind, key, extra = style
    if kind == "path":
        return f"{BASE_URL}{LIST_PATH}/page/{n}" if n > 1 else BASE_URL + LIST_PATH
    q = dict(extra)
    q[key] = n
    return f"{BASE_URL}{LIST_PATH}?{urlencode(q)}"


def crawl_listing_requests(max_pages=300, log=print):
    r = fetch(BASE_URL + LIST_PATH)
    html = r.text
    total = detect_total(html)
    first = links_from_html(html, r.url)
    if not first:
        log("   Halaman daftar tidak berisi link putusan di HTML-nya (kemungkinan dibangun JavaScript).")
        return None, total, html
    style = detect_page_style(html, r.url)
    log(f"   Halaman 1: {len(first)} link. Pola halaman: {page_url(2, style)}")
    items = {d["slug"]: {**d, "halaman_daftar": 1} for d in first}
    empty = 0
    bar = tqdm(range(2, max_pages + 1), desc="Halaman daftar")
    for n in bar:
        try:
            rr = fetch(page_url(n, style))
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 404:
                break
            raise
        new = [d for d in links_from_html(rr.text, rr.url) if d["slug"] not in items]
        if not new:
            empty += 1
            if empty >= 2:
                break
            continue
        empty = 0
        for d in new:
            items[d["slug"]] = {**d, "halaman_daftar": n}
        if hasattr(bar, "set_postfix"):
            bar.set_postfix(total=len(items))
    if hasattr(bar, "close"):
        bar.close()
    return list(items.values()), total, html


def paginate_api(api_url, log=print, max_pages=500):
    pr = urlparse(api_url)
    base = f"{pr.scheme}://{pr.netloc}{pr.path}"
    q = {k: v[0] for k, v in parse_qs(pr.query, keep_blank_values=True).items()}
    page_key = next((k for k in q if k.lower() in _PAGE_KEYS), None)
    off_key = next((k for k in q if k.lower() in ("offset", "skip", "start")), None)
    lim_key = next((k for k in q if k.lower() in ("limit", "per_page", "perpage", "pagesize", "page_size",
                                                   "size", "take", "length")), None)
    if not page_key and not off_key:
        page_key = "page"
    if lim_key:  # coba minta 100 data per halaman supaya lebih cepat
        try:
            test = dict(q, **{lim_key: 100})
            if off_key:
                test[off_key] = 0
            if page_key:
                test[page_key] = q.get(page_key, "1") if q.get(page_key) == "0" else 1
            n_test = len(best_records(fetch(base + "?" + urlencode(test)).json()))
            if n_test > int(q.get(lim_key) or 0):
                q[lim_key] = 100
        except Exception:
            pass
    per = int(q.get(lim_key) or 0) if lim_key else 0
    start = 0 if (page_key and q.get(page_key) == "0") else 1
    items, empty = {}, 0
    log(f"   Mengambil data dari API: {base}")
    bar = tqdm(range(max_pages), desc="Halaman API")
    for i in bar:
        qq = dict(q)
        if off_key:
            qq[off_key] = i * (per or len(items) or 10) if i else 0
        else:
            qq[page_key] = start + i
        recs = best_records(fetch(base + "?" + urlencode(qq)).json())
        new = 0
        for rec in recs:
            it = record_to_item(rec)
            if it["slug"] not in items:
                items[it["slug"]] = {**it, "halaman_daftar": i + 1}
                new += 1
        if hasattr(bar, "set_postfix"):
            bar.set_postfix(total=len(items))
        if not new:
            empty += 1
            if empty >= 2 or not recs:
                break
        else:
            empty = 0
        if off_key and not per:
            per = len(recs)
    if hasattr(bar, "close"):
        bar.close()
    return list(items.values())


# ------------------------------------------------- browser (Playwright) ----
def playwright_ready():
    try:
        import playwright  # noqa: F401
        return True
    except ImportError:
        return False


def _in_thread(fn, *a, **k):
    # Playwright sync API tidak boleh jalan di event loop Jupyter → jalankan di thread terpisah
    with _cf.ThreadPoolExecutor(1) as ex:
        return ex.submit(fn, *a, **k).result()


def _launch(p):
    kw = {"headless": True}
    exe = os.environ.get("PW_CHROMIUM_PATH")
    if exe:
        kw["executable_path"] = exe
    return p.chromium.launch(**kw)


def _pick_api(captured):
    best = None
    for url, data in captured:
        recs = best_records(data)
        if len(recs) >= 3:
            score = (("putusan" in url.lower()) * 2 + len(recs))
            if not best or score > best[0]:
                best = (score, url)
    return best[1] if best else None


_NEXT_SEL = ["a[rel='next']", "li.next:not(.disabled) a", "[aria-label*='next' i]", "[aria-label*='berikut' i]",
             "a:has-text('Selanjutnya')", "button:has-text('Selanjutnya')", "a:has-text('Next')",
             "button:has-text('Next')", "a:has-text('›')", "button:has-text('›')", "a:has-text('»')",
             "button:has-text('»')", "button:has-text('Muat lebih')", "button:has-text('Lihat lebih')",
             "button:has-text('Load more')"]


def _click_next(pg, n):
    cands = [f"a:text-is('{n}')", f"button:text-is('{n}')", f"li:text-is('{n}')"] + _NEXT_SEL
    for sel in cands:
        try:
            loc = pg.locator(sel)
            for i in range(min(loc.count(), 3)):
                el = loc.nth(i)
                if el.is_visible() and el.is_enabled() and el.get_attribute("disabled") is None:
                    el.click(timeout=5000)
                    return True
        except Exception:
            continue
    return False


def crawl_listing_browser(max_pages=300, log=print):
    def job():
        from playwright.sync_api import sync_playwright
        captured, items = [], {}
        with sync_playwright() as p:
            b = _launch(p)
            pg = b.new_page(user_agent=UA)

            def on_resp(resp):
                try:
                    if "json" in (resp.headers.get("content-type") or "") and resp.status == 200:
                        captured.append((resp.url, resp.json()))
                except Exception:
                    pass
            pg.on("response", on_resp)
            pg.goto(BASE_URL + LIST_PATH, wait_until="networkidle", timeout=90000)
            pg.wait_for_timeout(1500)
            for d in links_from_html(pg.content(), pg.url):
                items.setdefault(d["slug"], {**d, "halaman_daftar": 1})
            total = detect_total(pg.content())
            api = _pick_api(captured)
            if api:
                b.close()
                return "api", api, total, list(items.values())
            log(f"   Halaman 1 (browser): {len(items)} link.")
            # coba ?page=2 lewat URL dulu, kalau tidak jalan → klik tombol halaman
            url_mode = False
            pg.goto(page_url(2, ("query", "page", {})), wait_until="networkidle", timeout=90000)
            pg.wait_for_timeout(1000)
            new = [d for d in links_from_html(pg.content(), pg.url) if d["slug"] not in items]
            if new:
                url_mode = True
                for d in new:
                    items[d["slug"]] = {**d, "halaman_daftar": 2}
            else:
                pg.goto(BASE_URL + LIST_PATH, wait_until="networkidle", timeout=90000)
            empty = 0
            for n in tqdm(range(3 if url_mode else 2, max_pages + 1), desc="Halaman (browser)"):
                if url_mode:
                    pg.goto(page_url(n, ("query", "page", {})), wait_until="networkidle", timeout=90000)
                else:
                    pg.mouse.wheel(0, 20000)
                    if not _click_next(pg, n):
                        break
                    try:
                        pg.wait_for_load_state("networkidle", timeout=30000)
                    except Exception:
                        pass
                pg.wait_for_timeout(int(CFG.delay * 1000))
                api = _pick_api(captured)
                if api:
                    b.close()
                    return "api", api, total, list(items.values())
                new = [d for d in links_from_html(pg.content(), pg.url) if d["slug"] not in items]
                for d in new:
                    items[d["slug"]] = {**d, "halaman_daftar": n}
                empty = 0 if new else empty + 1
                if empty >= 2:
                    break
            b.close()
            return "dom", None, total, list(items.values())
    return _in_thread(job)


def browser_render_many(urls, on_done):
    """Render tiap URL di browser; on_done(url, html, captured_json) dipanggil per URL."""
    def job():
        from playwright.sync_api import sync_playwright
        with sync_playwright() as p:
            b = _launch(p)
            pg = b.new_page(user_agent=UA)
            captured = []

            def on_resp(resp):
                try:
                    if "json" in (resp.headers.get("content-type") or "") and resp.status == 200:
                        captured.append((resp.url, resp.json()))
                except Exception:
                    pass
            pg.on("response", on_resp)
            for u in tqdm(urls, desc="Detail (browser)"):
                captured.clear()
                try:
                    pg.goto(u, wait_until="networkidle", timeout=90000)
                    pg.wait_for_timeout(int(CFG.delay * 1000))
                    on_done(u, pg.content(), list(captured), None)
                except Exception as e:
                    on_done(u, None, [], e)
            b.close()
    _in_thread(job)


def collect_listing(mode="auto", max_pages=300, log=print):
    """Kembalikan DataFrame daftar putusan (slug, url, judul_daftar, ...)."""
    items, total = None, None
    if mode in ("auto", "requests"):
        log("🔎 Cara 1: membaca HTML halaman daftar langsung…")
        try:
            items, total, html = crawl_listing_requests(max_pages, log)
            if items is None:
                api = None
                for sc in BeautifulSoup(html, "lxml").find_all("script"):
                    txt = sc.string or ""
                    if sc.get("type") in ("application/json", "application/ld+json") or sc.get("id") == "__NEXT_DATA__":
                        try:
                            recs = best_records(json.loads(txt))
                            if len(recs) >= 3:
                                items = [record_to_item(r) for r in recs]
                                log(f"   Data ditemukan di JSON tertanam halaman: {len(items)} entri (hanya halaman 1).")
                                api = True
                        except Exception:
                            pass
                if api is None:
                    items = None
        except Exception as e:
            log(f"   Gagal: {e}")
            items = None
        if items and total and len(items) < total * 0.95:
            log(f"   ⚠️ Baru dapat {len(items)} dari ±{total} putusan dengan cara 1.")
            if mode == "auto" and playwright_ready():
                log("   Mencoba cara browser supaya lengkap…")
                try:
                    items_b = _listing_browser(max_pages, log)
                except Exception as e:
                    log(f"   Cara browser gagal ({e}); memakai hasil cara 1.")
                    items_b = None
                if items_b and len(items_b) > len(items):
                    items = items_b
        elif not items and mode == "auto":
            if playwright_ready():
                try:
                    items = _listing_browser(max_pages, log)
                except Exception as e:
                    log(f"❌ Cara browser gagal: {e}")
            else:
                log("❌ Situs butuh browser. Jalankan sel 'Pasang browser' (Langkah 1b) lalu ulangi langkah ini.")
    elif mode == "browser":
        items = _listing_browser(max_pages, log)
    if not items:
        return pd.DataFrame(columns=["slug", "url", "judul_daftar"])
    df = pd.DataFrame(items)
    meta = df["slug"].map(info_from_slug).apply(pd.Series)
    df = pd.concat([df, meta], axis=1)
    if total:
        log(f"ℹ️  Situs menyebut ±{total} putusan; terkumpul {len(df)}.")
    return df


def _listing_browser(max_pages, log):
    if not playwright_ready():
        log("❌ Playwright belum terpasang. Jalankan sel 'Langkah 1b' dulu.")
        return None
    log("🌐 Cara 2: membuka situs dengan browser otomatis (lebih lambat, tapi bisa membaca situs JavaScript)…")
    kind, api, total, items = crawl_listing_browser(max_pages, log)
    if kind == "api":
        log(f"   ✅ Situs memakai API JSON: {api}")
        try:
            got = paginate_api(api, log, max_pages)
            if got:
                return got
        except Exception as e:
            log(f"   API gagal dibaca langsung ({e}); pakai hasil browser saja.")
    return items


# --------------------------------------------------------------- detail ----
_LABEL_WORDS = ("nomor", "no.", "no ", "tahun", "jenis", "judul", "tanggal", "tgl", "tempat", "pemohon", "termohon",
                "status", "bidang", "subjek", "subyek", "bahasa", "sumber", "penetapan", "pengundangan", "singkatan",
                "lokasi", "kategori", "majelis", "amar", "abstrak", "klasifikasi", "komisioner", "panitera",
                "register", "lembaga", "peradilan", "urusan", "penandatangan", "t.e.u", "teu", "pihak", "badan publik",
                "putusan", "mediator", "hasil", "keterangan", "dibaca", "diunduh", "dilihat", "objek", "obyek",
                "kata kunci", "wilayah", "provinsi", "instansi", "pengarang", "penerbit", "catatan", "isi")


def _is_label(s):
    s = s.strip().rstrip(":").strip().lower()
    return 0 < len(s) <= 40 and any(s.startswith(w.strip()) for w in _LABEL_WORDS)


def _norm_key(k):
    return clean(k).rstrip(":").strip()


def parse_detail_html(html, url, captured=None, slug=None):
    soup = BeautifulSoup(html, "lxml")
    fields, pdfs = {}, []

    # 1) data JSON tertanam (Next.js, ld+json) atau JSON dari API (mode browser)
    blobs = []
    for sc in soup.find_all("script"):
        if sc.get("id") == "__NEXT_DATA__" or sc.get("type") in ("application/json", "application/ld+json"):
            try:
                blobs.append(json.loads(sc.string or ""))
            except Exception:
                pass
    for _, data in (captured or []):
        blobs.append(data)
    for blob in blobs:
        d = best_dict(blob, slug)
        if d:
            for k, v in flatten(d).items():
                fields.setdefault("data." + k, v)

    for tag in soup(["script", "style", "noscript", "nav", "footer", "header"]):
        tag.decompose()

    h1 = soup.find("h1")
    og = soup.find("meta", attrs={"property": "og:title"})
    judul = clean(h1.get_text(" ")) if h1 else ""
    if not judul and og and og.get("content"):
        judul = clean(og["content"])
    if not judul and soup.title:
        judul = clean(soup.title.get_text())

    # 2) tabel label | nilai
    for tr in soup.find_all("tr"):
        cells = [clean(c.get_text(" ")) for c in tr.find_all(["th", "td"], recursive=False)]
        k = v = None
        if len(cells) == 2:
            k, v = cells
        elif len(cells) == 3 and cells[1] in (":", ""):
            k, v = cells[0], cells[2]
        if k and len(k) <= 60 and not k.isdigit():
            fields.setdefault(_norm_key(k), v.lstrip(": ").strip())
    # 3) dl/dt/dd
    for dt in soup.find_all("dt"):
        dd = dt.find_next_sibling("dd")
        if dd:
            fields.setdefault(_norm_key(dt.get_text(" ")), clean(dd.get_text(" ")))
    # 4) baris teks "Label : nilai" atau label & nilai di baris berurutan
    main = soup.find("main") or soup.find("article") or soup.body or soup
    lines = [clean(x) for x in main.get_text("\n").split("\n")]
    lines = [x for x in lines if x]
    i = 0
    while i < len(lines):
        ln = lines[i]
        m = re.match(r"^([^:]{1,40}?)\s*:\s*(.+)$", ln)
        if m and _is_label(m.group(1)):
            fields.setdefault(_norm_key(m.group(1)), m.group(2).strip())
        elif _is_label(ln) and i + 1 < len(lines) and not _is_label(lines[i + 1]) and len(ln) < len(lines[i + 1]) + 40:
            nxt = lines[i + 1].lstrip(": ").strip()
            if nxt and _norm_key(ln) not in fields:
                fields[_norm_key(ln)] = nxt
                i += 1
        i += 1

    # 5) link PDF
    def add_pdf(u):
        if not u:
            return
        pq = parse_qs(urlparse(u).query)
        if "file" in pq:  # viewer pdf.js: ...viewer.html?file=xxx.pdf
            u = urljoin(url, unquote(pq["file"][0]))
        if u not in pdfs:
            pdfs.append(u)
    for tag in BeautifulSoup(html, "lxml").find_all(["a", "iframe", "embed", "object", "source"]):
        src = tag.get("href") or tag.get("src") or tag.get("data")
        if not src or src.startswith(("javascript:", "mailto:", "#")):
            continue
        u = urljoin(url, src)
        if re.search(r"\.pdf($|\?|#)", u, re.I) or ("/uploads/" in u and _same_host(u)):
            add_pdf(u.split("#")[0])
    for m in re.finditer(r"""["'(=]([^"'()\s<>]+?\.pdf)(?=["')&?#\s])""", html, re.I):
        add_pdf(_abs_pdf(m.group(1).replace("\\/", "/"), url))
    for k, v in fields.items():
        if isinstance(v, str) and re.search(r"\.pdf$", v, re.I) and " " not in v:
            add_pdf(_abs_pdf(v, url))

    # buang nilai yang sama persis dengan label / terlalu panjang
    fields = {k: v for k, v in fields.items() if k and v != "" and k.lower() != str(v).lower()}
    return {"judul": judul, "pdf_urls": " | ".join(pdfs), **fields}


def _cache_file(slug):
    return CFG.cache_dir / (safe_name(slug, 150) + ".json")


def _base_row(row):
    d = {k: row[k] for k in row.index if not str(k).startswith("_")}
    return {k: ("" if (isinstance(v, float) and pd.isna(v)) else v) for k, v in d.items()}


def _finish(row, parsed):
    out = {**_base_row(row), **parsed}
    title = parsed.get("judul") or row.get("judul_daftar", "")
    der = derive_from_title(title, row.get("slug", ""))
    for k, v in der.items():
        if v and not out.get(k):
            out[k] = v
    if not out.get("pdf_urls") and row.get("pdf_dari_daftar"):
        out["pdf_urls"] = row["pdf_dari_daftar"]
    return out


def thin(parsed):
    extra = [k for k in parsed if k not in ("judul", "pdf_urls")]
    return len(extra) < 2 and not parsed.get("pdf_urls")


def scrape_details(df_list, use_cache=True, mode="auto", log=print):
    """Ambil halaman detail tiap putusan. Hasil disimpan per-putusan di cache → bisa dilanjutkan kalau terputus."""
    results, todo = {}, []
    for _, row in df_list.iterrows():
        f = _cache_file(row["slug"])
        if use_cache and f.exists():
            try:
                results[row["slug"]] = json.loads(f.read_text(encoding="utf-8"))
                continue
            except Exception:
                pass
        todo.append(row)
    if results:
        log(f"♻️  {len(results)} putusan diambil dari cache (sudah pernah di-scrape).")
    if not todo:
        return pd.DataFrame([results[s] for s in df_list["slug"] if s in results])

    def save(row, parsed, err=None):
        rec = _finish(row, parsed)
        if err:
            rec["error"] = str(err)[:300]
        else:
            _cache_file(row["slug"]).write_text(json.dumps(rec, ensure_ascii=False), encoding="utf-8")
        results[row["slug"]] = rec

    use_browser = mode == "browser"
    if mode == "auto":
        # uji 1–2 halaman dulu: kalau HTML-nya kosong (situs JavaScript), pindah ke browser
        sample = [r for r in todo[:2] if r.get("url")]
        thin_count = 0
        for row in sample:
            try:
                p = parse_detail_html(fetch(row["url"]).text, row["url"], slug=row["slug"])
                thin_count += thin(p)
            except Exception:
                thin_count += 1
        if sample and thin_count == len(sample):
            if playwright_ready():
                log("🌐 Halaman detail dibangun JavaScript → memakai browser otomatis.")
                use_browser = True
            else:
                log("⚠️ Halaman detail tampak kosong tanpa browser. Jalankan Langkah 1b untuk hasil lengkap.")

    if use_browser:
        by_url = {r["url"]: r for r in todo if r.get("url")}

        def done(u, html, captured, err):
            row = by_url[u]
            if err or html is None:
                save(row, {}, err or "gagal render")
            else:
                save(row, parse_detail_html(html, u, captured, row["slug"]))
        browser_render_many(list(by_url), done)
        for r in todo:
            if not r.get("url"):
                save(r, {}, "url kosong")
    else:
        for row in tqdm(todo, desc="Detail putusan"):
            if not row.get("url"):
                save(row, {}, "url kosong")
                continue
            try:
                save(row, parse_detail_html(fetch(row["url"]).text, row["url"], slug=row["slug"]))
            except Exception as e:
                save(row, {}, e)
    errs = sum(1 for r in results.values() if r.get("error"))
    if errs:
        log(f"⚠️ {errs} putusan gagal diambil (kolom 'error'). Jalankan ulang sel ini untuk mencoba lagi yang gagal saja.")
    return pd.DataFrame([results[s] for s in df_list["slug"] if s in results])


# ----------------------------------------------------------------- simpan ----
FRONT_COLS = ["judul", "jenis_dokumen", "nomor_putusan", "tahun", "pemohon", "termohon", "url", "pdf_urls"]


def order_columns(cols):
    cols = list(dict.fromkeys(cols))
    front = [c for c in FRONT_COLS if c in cols]
    return front + [c for c in cols if c not in front]


def save_table(df, name="putusan_jdih_ki"):
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
    CFG.out_dir.mkdir(parents=True, exist_ok=True)
    csv_p = CFG.out_dir / f"{name}.csv"
    xlsx_p = CFG.out_dir / f"{name}.xlsx"
    df.to_csv(csv_p, index=False, encoding="utf-8-sig")
    x = df.copy()
    for c in x.columns:
        if x[c].dtype == object:
            x[c] = x[c].map(lambda v: ILLEGAL_CHARACTERS_RE.sub("", v)[:32000] if isinstance(v, str) else v)
    x.to_excel(xlsx_p, index=False)
    return csv_p, xlsx_p


def download_pdfs(df, folder="pdf", log=print):
    out = CFG.out_dir / folder
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    for _, r in tqdm(list(df.iterrows()), desc="Unduh PDF"):
        urls = [u.strip() for u in str(r.get("pdf_urls") or "").split("|") if u.strip()]
        got = []
        base = safe_name(r.get("nomor_putusan") or r.get("slug") or "dokumen", 90)
        for i, u in enumerate(urls, 1):
            p = out / (base + (f"_{i}" if len(urls) > 1 else "") + ".pdf")
            if p.exists() and p.stat().st_size > 1000:
                got.append(str(p))
                continue
            try:
                resp = fetch(u)
                if not resp.content.startswith(b"%PDF") and "pdf" not in resp.headers.get("content-type", ""):
                    continue
                p.write_bytes(resp.content)
                got.append(str(p))
            except Exception as e:
                log(f"   gagal: {u} ({e})")
        paths.append(" | ".join(got))
    return paths


def pdf_text(path, max_pages=None):
    from pypdf import PdfReader
    try:
        rd = PdfReader(path)
        pages = rd.pages if not max_pages else rd.pages[:max_pages]
        return "\n".join((pg.extract_text() or "") for pg in pages).strip()
    except Exception as e:
        return f"[gagal membaca PDF: {e}]"


print("✅ Mesin scraper siap. Lanjut ke Langkah 2.")

## Langkah 2 — Pengaturan
Atur pilihan di bawah (atau biarkan bawaan), **lalu lanjut ke Langkah 3**. Pengaturan dibaca saat sel berikutnya dijalankan.

- **Cara ambil** — biarkan *Otomatis*. *Pakai browser* hanya kalau Otomatis gagal (butuh Langkah 1b).
- **Jeda** — jarak antar-request. Jangan terlalu kecil supaya tidak diblokir server.
- **Lanjutkan dari hasil sebelumnya** — kalau dicentang, daftar & detail yang sudah pernah diambil dipakai lagi.
  Hilangkan centang kalau ingin mengambil ulang dari nol (mis. ada putusan baru di situs).

In [ ]:
import ipywidgets as W
from IPython.display import display, HTML
_st = {"description_width": "initial"}
w_mode = W.ToggleButtons(options=[("Otomatis", "auto"), ("Tanpa browser", "requests"), ("Pakai browser", "browser")],
                         value="auto", description="Cara ambil:", style=_st)
w_delay = W.FloatSlider(value=1.0, min=0.3, max=5.0, step=0.1, description="Jeda antar-request (detik):",
                        style=_st, layout=W.Layout(width="480px"))
w_out = W.Text(value="hasil_scraping", description="Nama folder hasil:", style=_st)
w_cache = W.Checkbox(value=True, description="Lanjutkan dari hasil sebelumnya (pakai cache)", style=_st)
w_maxpages = W.BoundedIntText(value=300, min=1, max=5000, description="Batas halaman daftar yang dibuka:", style=_st)
display(W.VBox([w_mode, w_delay, w_out, w_cache, w_maxpages]))

## Langkah 3 — Ambil daftar semua putusan
Mengumpulkan judul & link semua putusan dari halaman daftar (belum isi detailnya).
Daftar disimpan ke `hasil_scraping/daftar_putusan.csv`. Situs menyebut ±904 putusan; jumlah yang terkumpul
ditampilkan di akhir.

In [ ]:
CFG.delay = w_delay.value
CFG.out_dir = Path(w_out.value.strip() or "hasil_scraping")
CFG.out_dir.mkdir(parents=True, exist_ok=True)
_list_file = CFG.out_dir / "daftar_putusan.csv"

if w_cache.value and _list_file.exists():
    DAFTAR = pd.read_csv(_list_file, dtype=str).fillna("")
    print(f"♻️  Memakai daftar tersimpan ({len(DAFTAR)} putusan). "
          "Hilangkan centang 'Lanjutkan…' di Langkah 2 untuk mengambil ulang.")
else:
    DAFTAR = collect_listing(w_mode.value, w_maxpages.value).fillna("")
    if len(DAFTAR):
        DAFTAR.to_csv(_list_file, index=False, encoding="utf-8-sig")

if len(DAFTAR) == 0:
    print("❌ Tidak ada putusan yang terkumpul. Lihat bagian 'Kalau ada masalah' di bawah notebook.")
else:
    for _c, _f in (("tahun_slug", lambda s: info_from_slug(s)["tahun_slug"]), ("jenis_slug", lambda s: info_from_slug(s)["jenis_slug"])):
        if _c not in DAFTAR:
            DAFTAR[_c] = DAFTAR["slug"].map(_f)
    print(f"✅ {len(DAFTAR)} putusan terkumpul.\n")
    print("Jumlah per tahun:")
    display(DAFTAR["tahun_slug"].value_counts().sort_index(ascending=False).to_frame("jumlah").T)
    print("Jumlah per jenis:")
    display(DAFTAR["jenis_slug"].value_counts().to_frame("jumlah").T)
    display(DAFTAR[["judul_daftar", "url"]].head(10))

## Langkah 4 — Saring putusan yang mau di-scrape
Biarkan kosong untuk mengambil **semua**. Tahun & jenis bisa pilih lebih dari satu (tahan `Ctrl`/`Cmd` sambil klik).
Kata kunci dicari di judul (semua kata harus ada, tidak peduli huruf besar/kecil), contoh: `pajak` atau `bank indonesia`.
Jumlah yang akan diambil langsung terlihat di bawah. Setelah itu lanjut ke Langkah 5.

In [ ]:
_years = sorted(set(DAFTAR["tahun_slug"]), reverse=True)
_jenis = sorted(set(DAFTAR["jenis_slug"]))
w_tahun = W.SelectMultiple(options=_years, description="Tahun:", rows=min(10, max(3, len(_years))), style=_st)
w_jenis = W.SelectMultiple(options=_jenis, description="Jenis:", rows=min(5, max(2, len(_jenis))), style=_st)
w_kata = W.Text(placeholder="mis. pajak", description="Kata kunci di judul:", style=_st, layout=W.Layout(width="480px"))
w_kecuali = W.Text(placeholder="mis. penetapan", description="Kecualikan kata:", style=_st, layout=W.Layout(width="480px"))
w_urut = W.Dropdown(options=[("Sesuai urutan situs", "situs"), ("Tahun terbaru dulu", "baru"), ("Tahun terlama dulu", "lama")],
                    description="Urutan:", style=_st)
w_n = W.BoundedIntText(value=0, min=0, max=max(1, len(DAFTAR)), description="Batas jumlah (0 = semua):", style=_st)
_lbl = W.HTML()

def apply_filter():
    d = DAFTAR.copy()
    if w_tahun.value:
        d = d[d["tahun_slug"].isin(w_tahun.value)]
    if w_jenis.value:
        d = d[d["jenis_slug"].isin(w_jenis.value)]
    teks = (d["judul_daftar"].astype(str) + " " + d["slug"].astype(str).str.replace("-", " ")).str.lower()
    for kata in w_kata.value.lower().split():
        d, teks = d[teks.str.contains(kata, regex=False)], teks[teks.str.contains(kata, regex=False)]
    for kata in w_kecuali.value.lower().split():
        d, teks = d[~teks.str.contains(kata, regex=False)], teks[~teks.str.contains(kata, regex=False)]
    if w_urut.value != "situs":
        d = d.assign(_t=pd.to_numeric(d["tahun_slug"], errors="coerce")).sort_values(
            "_t", ascending=(w_urut.value == "lama"), kind="stable").drop(columns="_t")
    if w_n.value:
        d = d.head(w_n.value)
    return d

def _refresh(*_):
    _lbl.value = f"<h3>➡️ {len(apply_filter())} dari {len(DAFTAR)} putusan akan di-scrape</h3>"
for _w in (w_tahun, w_jenis, w_kata, w_kecuali, w_urut, w_n):
    _w.observe(_refresh, "value")
_refresh()
display(W.VBox([W.HBox([w_tahun, w_jenis]), w_kata, w_kecuali, w_urut, w_n, _lbl]))

## Langkah 5 — Pratinjau & pilih kolom
Sel ini mengambil **3 putusan pertama** dari hasil saringan supaya terlihat data apa saja yang tersedia di situs.
Lalu **centang kolom yang mau disimpan**. Kolom `judul`, `nomor_putusan`, `tahun`, `pemohon`, `termohon`, `jenis_dokumen`
juga dibaca otomatis dari judul putusan.

In [ ]:
TERPILIH = apply_filter()
print(f"Putusan terpilih: {len(TERPILIH)}")
PREVIEW = scrape_details(TERPILIH.head(3), use_cache=w_cache.value, mode=w_mode.value)

_HELPER = {"slug", "halaman_daftar", "jenis_slug", "tahun_slug", "pdf_dari_daftar", "judul_daftar", "error"}
_cols = [c for c in order_columns(PREVIEW.columns) if c not in ("error",)]
print("\nContoh isi (kolom = 3 putusan pertama):")
with pd.option_context("display.max_colwidth", 80, "display.max_rows", 200):
    display(PREVIEW[_cols].T)

CB = {c: W.Checkbox(value=(c not in _HELPER and not c.startswith("api.")) or c == "slug",
                    description=c, indent=False, layout=W.Layout(width="320px")) for c in _cols}
w_baru = W.Checkbox(value=True, description="Ikutkan juga kolom baru yang mungkin muncul di putusan lain", style=_st)
_all = W.Button(description="Centang semua"); _none = W.Button(description="Kosongkan semua")
_all.on_click(lambda _: [setattr(cb, "value", True) for cb in CB.values()])
_none.on_click(lambda _: [setattr(cb, "value", False) for cb in CB.values()])
display(HTML("<h3>Pilih kolom yang disimpan:</h3>"),
        W.HBox([_all, _none]),
        W.GridBox(list(CB.values()), layout=W.Layout(grid_template_columns="repeat(auto-fill, 330px)")),
        w_baru)

## Langkah 6 — Scrape detail semua putusan terpilih
Ini langkah yang paling lama (±1–2 detik per putusan). Ada progress bar.
Kalau terputus atau ada yang gagal, **jalankan ulang sel ini** — yang sudah berhasil tidak diambil lagi.

In [ ]:
HASIL = scrape_details(TERPILIH, use_cache=True, mode=w_mode.value)
HASIL["error"] = HASIL["error"].fillna("") if "error" in HASIL else ""
_ok = int((HASIL["error"] == "").sum())
print(f"✅ Selesai: {_ok} berhasil, {len(HASIL) - _ok} gagal, dari {len(TERPILIH)} putusan.")
if "pdf_urls" in HASIL:
    print(f"📄 {(HASIL['pdf_urls'].fillna('') != '').sum()} putusan punya link PDF.")

## Langkah 7 — Simpan ke Excel & CSV
Hasil disimpan ke folder hasil sebagai `putusan_jdih_ki.xlsx` dan `putusan_jdih_ki.csv`
(CSV memakai UTF-8 dengan BOM supaya rapi dibuka di Excel).

In [ ]:
_pilih = [c for c, cb in CB.items() if cb.value]
if w_baru.value:
    _pilih += [c for c in HASIL.columns if c not in CB and c not in _HELPER and not c.startswith("api.")]
if (HASIL["error"] != "").any():
    _pilih.append("error")
_pilih = [c for c in order_columns(_pilih) if c in HASIL.columns]
FINAL = HASIL.reindex(columns=_pilih).fillna("")
FINAL.insert(0, "no", range(1, len(FINAL) + 1))
_csv, _xlsx = save_table(FINAL)
print(f"💾 Tersimpan:\n   {_xlsx}\n   {_csv}\n   ({len(FINAL)} baris × {len(FINAL.columns)} kolom)")
display(FINAL.head(10))

## Langkah 8 — (Opsional) Unduh PDF & ekstrak teks
- **Unduh PDF** — PDF putusan disimpan ke `hasil_scraping/pdf/`, nama file = nomor putusan. File yang sudah ada dilewati.
  ±904 PDF bisa beberapa ratus MB.
- **Ekstrak teks** — isi PDF dibaca jadi teks: kolom `teks_pdf` di Excel (dipotong 32.000 karakter karena batas Excel)
  dan teks lengkap per putusan di `hasil_scraping/teks/`. PDF hasil scan (gambar) tidak punya teks; hasilnya kosong.

Centang yang diinginkan, lalu jalankan sel **berikutnya**.

In [ ]:
w_pdf = W.Checkbox(value=False, description="Unduh file PDF", style=_st)
w_txt = W.Checkbox(value=False, description="Ekstrak teks PDF (otomatis mengunduh PDF)", style=_st)
w_txt_pages = W.BoundedIntText(value=0, min=0, max=500, description="Halaman PDF yang dibaca (0 = semua):", style=_st)
display(W.VBox([w_pdf, w_txt, w_txt_pages]))

In [ ]:
if not (w_pdf.value or w_txt.value):
    print("Dilewati (tidak ada yang dicentang).")
else:
    if "pdf_urls" not in FINAL:
        FINAL["pdf_urls"] = HASIL["pdf_urls"].fillna("").values if "pdf_urls" in HASIL else ""
    FINAL["file_pdf_lokal"] = download_pdfs(FINAL.assign(slug=HASIL["slug"].values))
    print(f"📥 {(FINAL['file_pdf_lokal'] != '').sum()} dari {len(FINAL)} putusan berhasil diunduh PDF-nya.")
    if w_txt.value:
        _tdir = CFG.out_dir / "teks"; _tdir.mkdir(exist_ok=True)
        _teks = []
        for _paths in tqdm(FINAL["file_pdf_lokal"], desc="Ekstrak teks"):
            _t = "\n\n".join(pdf_text(p, w_txt_pages.value or None) for p in _paths.split(" | ") if p)
            for p in _paths.split(" | "):
                if p:
                    (_tdir / (Path(p).stem + ".txt")).write_text(_t, encoding="utf-8")
            _teks.append(_t)
        FINAL["teks_pdf"] = _teks
        print(f"📝 Teks tersimpan di {_tdir}")
    _csv, _xlsx = save_table(FINAL)
    print(f"💾 Excel/CSV diperbarui: {_xlsx}")

## Langkah 9 — (Opsional) Kemas jadi ZIP & unduh
Di Google Colab file langsung terunduh ke komputer. Di Jupyter lokal, file ZIP ada di folder yang sama dengan notebook.

In [ ]:
import shutil
_zip = shutil.make_archive(str(CFG.out_dir), "zip", root_dir=CFG.out_dir, base_dir=".")
print(f"🗜️  {_zip}")
try:
    from google.colab import files
    files.download(_zip)
except ImportError:
    pass

## 🛠️ Kalau ada masalah

| Gejala | Solusi |
|---|---|
| `ModuleNotFoundError` | Jalankan Langkah 1 lalu Langkah 0 lagi. |
| `NameError: name 'w_...' is not defined` / `'DAFTAR'` | Ada sel sebelumnya yang belum dijalankan. Jalankan dari atas secara urut. |
| Pesan *"Jalankan Langkah 1b"* / daftar kosong | Situs butuh browser. Jalankan Langkah 1b → Langkah 0 → ulangi Langkah 3. |
| Jumlah terkumpul jauh di bawah ±904 | Di Langkah 2 pilih **Pakai browser**, hilangkan centang *Lanjutkan…*, lalu ulangi Langkah 3. |
| Banyak error `429` / `503` / timeout | Server sedang sibuk atau membatasi. Naikkan **Jeda** di Langkah 2 (mis. 3 detik), tunggu sebentar, jalankan ulang Langkah 6. |
| `ConnectionError` / tidak bisa membuka situs | Cek apakah situs bisa dibuka di browser biasa. Kalau situs memblokir server luar negeri (mis. Colab), jalankan notebook di komputer sendiri (Jupyter/VS Code/Anaconda). |
| Kolom detail banyak yang kosong | Isi tiap putusan di situs memang tidak seragam. Untuk isi lengkapnya, pakai Langkah 8 (teks PDF). |
| Ingin ambil ulang dari nol | Hapus folder `hasil_scraping` atau hilangkan centang *Lanjutkan…* di Langkah 2. |

**Etika:** data JDIH adalah informasi publik, tapi tetap gunakan jeda yang wajar dan jangan menjalankan
banyak salinan scraper sekaligus.